In [8]:
import cv2, time, numpy as np
from fastdtw import fastdtw
from scipy.spatial.distance import cdist
from tqdm import tqdm
from ultralytics import YOLO as YOLO8
from ultralytics import YOLO as YOLO
import tensorflow_hub as hub
import tensorflow as tf


# -------------------------------------------------------
#  Model loading
# -------------------------------------------------------
model_neuroyolo = YOLO(f'best.pt') 
model_yolo8 = YOLO8("yolov8n-pose.pt") 
movenet = hub.load("https://tfhub.dev/google/movenet/singlepose/thunder/4")

In [10]:
# -------------------------------------------------------
#  Inference wrappers
# -------------------------------------------------------

def infer_yolo(model, frame):
    results = model.predict(frame, verbose=False)
    keypoints = results[0].keypoints.xy.cpu().numpy() if results[0].keypoints is not None else np.zeros((1,17,2))
    return keypoints

def infer_yolo8(frame):
    r = model_yolo8.predict(frame, verbose=False)
    if r and r[0].keypoints is not None:
        kp = r[0].keypoints.xy.cpu().numpy()  # (n,17,2) for detected persons
        kp = kp[:1] if kp.shape[0] > 1 else kp  # take first person
        return kp
    return np.zeros((1,17,2))

def infer_movenet(frame):
    """Single-person MoveNet inference returning (1,17,2) keypoints in pixel coords."""
    # Preprocess
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (256, 256))
    # MoveNet expects int32 tensor [1,256,256,3]
    input_tensor = tf.convert_to_tensor(img[np.newaxis, ...], dtype=tf.int32)

    # Inference
    outputs = movenet.signatures['serving_default'](input_tensor)
    keypoints = outputs['output_0'].numpy()[0, 0, :, :2]   # (17,2) normalized 0–1

    # Rescale back to original image size
    h, w, _ = frame.shape
    keypoints[:, 0] *= w
    keypoints[:, 1] *= h

    return keypoints[None, :, :]                           # (1,17,2)

# -------------------------------------------------------
#  Metric functions (same as before)
# -------------------------------------------------------

def compute_jitter_variance(keypoints):
    if len(keypoints) < 2:
        return np.nan
    diffs = np.diff(keypoints, axis=0)
    return np.var(np.linalg.norm(diffs, axis=2))

def compute_msi(keypoints):
    if keypoints.shape[1] < 3: return np.nan
    A, B, C = keypoints[:,5,:], keypoints[:,6,:], keypoints[:,7,:]
    v1, v2 = A-B, C-B
    cos = np.sum(v1*v2,axis=1)/(np.linalg.norm(v1,axis=1)*np.linalg.norm(v2,axis=1)+1e-6)
    ang = np.degrees(np.arccos(np.clip(cos,-1,1)))
    return np.mean(np.abs(np.diff(ang)))

def align_keypoints_dim(kp, target_dim=17):
    """Ensures both models output the same joint dimensionality."""
    if kp.shape[1] == target_dim:
        return kp
    elif kp.shape[1] > target_dim:
        # keep first 17 joints (COCO-compatible subset)
        return kp[:, :target_dim, :]
    else:
        # pad with zeros if fewer
        pad = np.zeros((kp.shape[0], target_dim - kp.shape[1], 2))
        return np.concatenate([kp, pad], axis=1)
    
def normalize_pose(kp):
    """Normalize by torso length to remove scale bias."""
    if kp.size == 0:
        return kp
    # shoulders and hips indices for COCO (5,6,11,12)
    torso = np.linalg.norm(kp[:,5,:] - kp[:,11,:], axis=1).mean() + 1e-6
    return kp / torso

def compute_psm(kp1, kp2):
    """ Pose Similarity Metric using DTW between flattened joint coordinates """
    # add before DTW
    kp1 = normalize_pose(align_keypoints_dim(kp1))
    kp2 = normalize_pose(align_keypoints_dim(kp2))

    kp1 = align_keypoints_dim(kp1)
    kp2 = align_keypoints_dim(kp2)
    n = min(len(kp1), len(kp2))
    s1, s2 = kp1[:n].reshape(n, -1), kp2[:n].reshape(n, -1)
    dist, _ = fastdtw(s1, s2)
    return max(0, 100 - dist / max(len(s1), len(s2)))

# -------------------------------------------------------
#  Evaluation routine
# -------------------------------------------------------

def run_video(video_path, model_func, label):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(frame)
    cap.release()

    # inference & timing
    t0 = time.time()
    all_kp = []
    for f in tqdm(frames, desc=label):
        kp = model_func(f)
        all_kp.append(kp)
    avg_ms = (time.time()-t0)/len(frames)*1000
    return np.vstack(all_kp), avg_ms

In [7]:
if __name__ == "__main__":

    video_participant = "videos\C1_6.mp4"
    video_coach = "videos\P1_6.mp4"

    # Run models
    kp_yolo8,  t_y8  = run_video(video_participant, infer_yolo8,        "YOLOv8n-pose")
    kp_movenet, t_mv = run_video(video_participant, infer_movenet, "MoveNet-Thunder")


    # Reference
    kp_coach, _ = run_video(video_coach, lambda f: infer_yolo(model_neuroyolo,f), "Coach Reference")

    # Metrics
    results = []
    # Then include in results aggregation:
    for name, kp, t in [("YOLOv8n-pose", kp_yolo8, t_y8),
                        ("MoveNet-Thunder", kp_movenet, t_mv)]:
        jitter = compute_jitter_variance(kp)
        msi    = compute_msi(kp)
        psm    = compute_psm(kp, kp_coach)  # your aligned/normalized version
        results.append((name, t, jitter, msi, psm))

    print("\n--- Comparison Results ---")
    print(f"{'Model':<22}{'ms/frame':>10}{'Jitter Var':>15}{'MSI':>10}{'PSM':>10}")
    for name, t, j, m, p in results:
        print(f"{name:<22}{t:>10.2f}{j:>15.3f}{m:>10.2f}{p:>10.2f}")


Coach Reference: 100%|██████████| 1800/1800 [02:10<00:00, 13.75it/s]



--- Comparison Results ---
Model                   ms/frame     Jitter Var       MSI       PSM
YOLOv8n-pose               78.82          7.897      0.33     58.54
MoveNet-Thunder            24.08         15.370      1.10     17.50
